<a href="https://colab.research.google.com/github/Lupo6789/tesi_scacchi/blob/main/02_Feature_Engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In questa prima fase del Feature Engineering, vengono caricati separatamente i dataset di addestramento e valutazione, mantenendo la rigida separazione stabilita nel Notebook 01.

In [11]:
import pandas as pd
import numpy as np

print("1. CARICAMENTO E PREPARAZIONE DATI")

# 1. Definizione della formula ufficiale Lichess
# Il formato di 'increment_code' è "Minuti+Secondi" (es. "10+5")
# Formula Lichess: (MinutiBase * 60) + (Incremento * 40)
def stima_tempo_lichess(increment_code):
    try:
        base_min, inc_sec = increment_code.split('+')
        tempo_totale_sec = (int(base_min) * 60) + (int(inc_sec) * 40)
        return tempo_totale_sec
    except Exception as e:
        return np.nan

# 2. Caricamento dei due dataset separati dal Notebook 01
# Questo garantisce che il Test Set subisca le stesse trasformazioni ma non si mescoli
df_train = pd.read_csv('train_cleaned.csv')
df_test = pd.read_csv('test_cleaned.csv')

# 3. Calcolo del tempo stimato
df_train['tempo_stimato_sec'] = df_train['increment_code'].apply(stima_tempo_lichess)
df_test['tempo_stimato_sec'] = df_test['increment_code'].apply(stima_tempo_lichess)

#Calcolo della lunghezza reale della partita
#Ci serve contare gli spazi nella stringa 'moves' per avere il numero di semimosse
df_train['numero_mosse_reali'] = df_train['moves'].apply(lambda x: len(str(x).split()))
df_test['numero_mosse_reali'] = df_test['moves'].apply(lambda x: len(str(x).split()))

print(f"Train Set preparato: {df_train.shape[0]} partite.")
print(f"Test Set preparato: {df_test.shape[0]} partite.")
print("Feature 'tempo_stimato_sec' e 'numero_mosse_reali' calcolate con successo.")

1. CARICAMENTO E PREPARAZIONE DATI
Train Set preparato: 14580 partite.
Test Set preparato: 3645 partite.
Feature 'tempo_stimato_sec' e 'numero_mosse_reali' calcolate con successo.


 Per ogni soglia N, vengono mantenute esclusivamente le partite con numero_mosse_reali > N. Si escludono quindi a priori sia le partite troppo brevi, sia quelle concluse esattamente alla mossa N (le quali conterranno già l'esito finale intrinseco, come lo scacco matto). Questo elimina l'utilizzo di informazioni provenienti "dal futuro".
 Mobilità Pseudo-Legale: Il conteggio della mobilità complessiva utilizza pseudo_legal_moves al posto delle mosse puramente legali. Questo previene errori computazionali nelle casistiche in cui, forzando artificialmente l'analisi di un giocatore fuori dal proprio turno, il suo Re si trovi sotto attacco (generando una scacchiera temporaneamente illegale per la logica del motore). Viene definitivamente cancellata assieme a numero_mosse_reali, impedendo che il modello possa vedere nel futuro la conclusione della partita.

In [12]:
!pip install python-chess
import chess

print("INIZIO ESTRAZIONE POSIZIONALE (FEATURE ENGINEERING)")

#Definiamo la funzione "motore" che scatta la fotografia alla mossa N
def estrai_features_posizionali(moves_string, n_semimosse):
    board = chess.Board()
    mosse = str(moves_string).split()

    # Avanziamo la scacchiera virtuale fino alla mossa N
    for i in range(min(n_semimosse, len(mosse))):
        board.push_san(mosse[i])

    # Vantaggio Materiale
    valori = {chess.PAWN: 1, chess.KNIGHT: 3, chess.BISHOP: 3, chess.ROOK: 5, chess.QUEEN: 9}
    mat_b = sum(valori.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.WHITE)
    mat_n = sum(valori.get(p.piece_type, 0) for p in board.piece_map().values() if p.color == chess.BLACK)
    vantaggio_materiale = mat_b - mat_n

    # 2. Mobilità Pseudo-Legale
    mob_b = sum(1 for move in board.pseudo_legal_moves if board.color_at(move.from_square) == chess.WHITE)
    mob_n = sum(1 for move in board.pseudo_legal_moves if board.color_at(move.from_square) == chess.BLACK)
    vantaggio_mobilita = mob_b - mob_n

    # 3. Diritti di Arrocco e Scacco
    arr_b = int(board.has_castling_rights(chess.WHITE))
    arr_n = int(board.has_castling_rights(chess.BLACK))
    scacco_b = 1 if board.is_check() and board.turn == chess.WHITE else 0
    scacco_n = 1 if board.is_check() and board.turn == chess.BLACK else 0
    # 4. Sviluppo Pezzi Leggeri
    sviluppo_b = sum(1 for sq in board.pieces(chess.KNIGHT, chess.WHITE) if sq not in [chess.B1, chess.G1]) + \
                 sum(1 for sq in board.pieces(chess.BISHOP, chess.WHITE) if sq not in [chess.C1, chess.F1])
    sviluppo_n = sum(1 for sq in board.pieces(chess.KNIGHT, chess.BLACK) if sq not in [chess.B8, chess.G8]) + \
                 sum(1 for sq in board.pieces(chess.BISHOP, chess.BLACK) if sq not in [chess.C8, chess.F8])
    vantaggio_sviluppo = sviluppo_b - sviluppo_n

    return vantaggio_materiale, vantaggio_mobilita, arr_b, arr_n, scacco_b, scacco_n, vantaggio_sviluppo

#Soglie temporali da analizzare
soglie = [10, 20, 30, 40]

#Il nostro approccio a doppio binario per mantenere il Train/Test split immacolato
datasets = {'train': df_train, 'test': df_test}

for split_name, df_current in datasets.items():
    print(f"\n--- ELABORAZIONE DEL {split_name.upper()} SET")

    for N in soglie:
        # 1. Filtro dinamico > N
        df_N = df_current[df_current['numero_mosse_reali'] > N].copy()
        print(f"Estrazione N={N}: restano {df_N.shape[0]} partite in corso valide.")

        # 2. Estrazione delle feature
        risultati = df_N['moves'].apply(lambda x: estrai_features_posizionali(x, N))
        col_names = [f'mat_{N}', f'mob_{N}', f'arr_b_{N}', f'arr_n_{N}', f'scacco_b_{N}', f'scacco_n_{N}', f'sviluppo_{N}']
        df_N[col_names] = pd.DataFrame(risultati.tolist(), index=df_N.index)

        # 3. Rimozione definitiva del Data Leakage
        df_N = df_N.drop(columns=['moves', 'numero_mosse_reali'])

        # 4. Salvataggio del file specifico
        nome_file = f"{split_name}_features_N{N}.csv"
        df_N.to_csv(nome_file, index=False)

print("\nOPERAZIONE COMPLETATA")
print("I file per ogni step N sono stati creati e sono pronti per i modelli!")

INIZIO ESTRAZIONE POSIZIONALE (FEATURE ENGINEERING)

--- ELABORAZIONE DEL TRAIN SET
Estrazione N=10: restano 14107 partite in corso valide.
Estrazione N=20: restano 13279 partite in corso valide.
Estrazione N=30: restano 11967 partite in corso valide.
Estrazione N=40: restano 10135 partite in corso valide.

--- ELABORAZIONE DEL TEST SET
Estrazione N=10: restano 3521 partite in corso valide.
Estrazione N=20: restano 3340 partite in corso valide.
Estrazione N=30: restano 3018 partite in corso valide.
Estrazione N=40: restano 2617 partite in corso valide.

OPERAZIONE COMPLETATA
I file per ogni step N sono stati creati e sono pronti per i modelli!


Feature aggiunte = tempo stimato in secondi, mat_N: la differenza tra punteggio materiale bianco - materiale nero; mobN= LA differenza tra le mosse pseudolegali del bianco e del nero; arr b N e arr n N: indicano se il bianco e nero abbiano il diritto di arroccare; scacco_b_N e scacco_n_N: booleani che indicano se il bianco o nero siano sotto scacco e sviluppo_ N, ovvero la differenza di sviluppo tra i pezzi bianchi e i neri.